# Scoring benchmark

Score `P(Y = y | x)`, the probability of the pattern each held-out row actually has, from one fitted
model. Data are synthetic: 6 features, linear margins, 3,000 training rows, 100 scored rows per d, and two true Σ:
**equi**, equicorrelated at ρ = 0.4 (well-conditioned), and **pairwise**, a rank-2 factor plus noise projected
to the nearest correlation matrix, which sits on the PSD boundary as a real pairwise fit does. The model is `MultivariateProbit(inner="linear", dependence="pairwise")`.

Every column scores that same fitted model (η̂, Σ̂), except statsmodels, which fits one `Probit` per
outcome and has no Σ, so its pattern probability is a product of marginals.

**Reference:** `scipy.stats.multivariate_normal.cdf` with `abseps=1e-7, releps=1e-6`, one row at a time.
**Error:** absolute difference from the reference, median and max over rows. **Time:** wall-clock per row,
after one warm-up call; statsmodels is timed on prediction only.

The logic is in `bench.py`. To include the keyed orthant evaluator at `resolution="high"`, put a key in
`$ORTHANT_KEY` or `~/.orthant/key` and rerun; without one it runs only at d ≤ 3 and `resolution="low"`.
R `mvProbit` is not run here.

In [1]:
import warnings, platform, os
import pandas as pd
import bench
warnings.filterwarnings("ignore")
print("orthant keyed:", bench.ORTHANT_KEYED, "| pybhatlib:", bench.mvncd_batch is not None,
      "| statsmodels:", bench.sm is not None)
print(platform.processor() or platform.machine(), os.cpu_count(), "cores")

orthant keyed: True | pybhatlib: True | statsmodels: True
x86_64 2 cores


In [2]:
rows, info = [], {}
for sigma in ("equi", "pairwise"):
    for d in (3, 5, 8, 12, 16):
        r, median_p, min_eig = bench.run_cell(d, sigma=sigma)
        rows += r
        info[(sigma, d)] = dict(median_p=round(median_p, 4), fitted_min_eig=f"{min_eig:.1e}")
df = pd.DataFrame(rows)
df.to_csv("results.csv", index=False)
pd.DataFrame(info).T

median_p fitted_min_eig
equi     3     0.348        5.4e-01
         5     0.154        4.9e-01
         8    0.1005        4.5e-01
         12   0.0221        4.3e-01
         16   0.0024        4.2e-01
pairwise 3     0.431        2.2e-01
         5     0.171        3.3e-03
         8    0.0974        1.0e-08
         12   0.0249        1.0e-08
         16   0.0131        1.0e-08

In [3]:
def cell(r):
    t = f"{r.s_per_row:.1e} s"
    return t if r.method == "reference" else f"{t} / {r.median_err:.4f} ({r.max_err:.4f})"

table = df.assign(cell=df.apply(cell, axis=1)).pivot(index=["sigma", "d"], columns="method", values="cell").fillna("—")
table

method                     SciPy default           mvp orthant (high)  \
sigma    d                                                              
equi     3   4.2e-03 s / 0.0000 (0.0000)  9.5e-06 s / 0.0001 (0.0002)   
         5   6.6e-03 s / 0.0000 (0.0000)  1.3e-05 s / 0.0001 (0.0003)   
         8   1.3e-02 s / 0.0000 (0.0000)  2.6e-05 s / 0.0001 (0.0005)   
         12  1.9e-02 s / 0.0000 (0.0000)  4.1e-05 s / 0.0001 (0.0007)   
         16  2.6e-02 s / 0.0000 (0.0000)  6.6e-05 s / 0.0000 (0.0006)   
pairwise 3   6.5e-03 s / 0.0000 (0.0000)  9.2e-06 s / 0.0008 (0.0029)   
         5   1.1e-01 s / 0.0000 (0.0000)  1.3e-05 s / 0.0061 (0.0338)   
         8   8.8e-01 s / 0.0000 (0.0000)  2.5e-05 s / 0.0048 (0.0379)   
         12  1.2e+00 s / 0.0000 (0.0000)  3.9e-05 s / 0.0034 (0.0431)   
         16  1.8e+00 s / 0.0000 (0.0000)  5.4e-05 s / 0.0037 (0.0453)   

method                    mvp quadrature               pybhatlib ovus  \
sigma    d                                                              
equi     3   6.2e-05 s / 0.0000 (0.0001)  6.5e-05 s / 0.0001 (0.0009)   
         5   3.3e-02 s / 0.0000 (0.0001)  9.4e-05 s / 0.0004 (0.0019)   
         8                             —  1.4e-04 s / 0.0003 (0.0023)   
         12                            —  1.9e-04 s / 0.0002 (0.0036)   
         16                            —  2.6e-04 s / 0.0000 (0.0011)   
pairwise 3   6.0e-05 s / 0.0000 (0.0001)  6.4e-05 s / 0.0004 (0.0047)   
         5   3.5e-02 s / 0.0000 (0.0001)  9.5e-05 s / 0.0015 (0.0276)   
         8                             —  1.4e-04 s / 0.0015 (0.0103)   
         12                            —  2.0e-04 s / 0.0007 (0.0139)   
         16                            —  2.5e-04 s / 0.0010 (0.0151)   

method       reference                  statsmodels  
sigma    d                                           
equi     3   2.5e-01 s  5.4e-07 s / 0.0284 (0.0992)  
         5   5.3e-01 s  6.4e-07 s / 0.0246 (0.1287)  
         8   1.4e+00 s  1.2e-06 s / 0.0145 (0.1154)  
         12  1.6e+00 s  1.2e-06 s / 0.0063 (0.0776)  
         16  1.1e+00 s  2.3e-06 s / 0.0015 (0.0536)  
pairwise 3   2.4e-01 s  4.2e-07 s / 0.0371 (0.1600)  
         5   1.3e+00 s  5.7e-07 s / 0.0523 (0.2549)  
         8   4.4e+00 s  9.3e-07 s / 0.0249 (0.1426)  
         12  9.5e+00 s  1.2e-06 s / 0.0143 (0.1855)  
         16  2.0e+01 s  1.5e-06 s / 0.0107 (0.1848)